# Étteremértékelések elemzése – 02. notebook
## Multi-label aspect detection DistilBERT finomhangolással

**Projekt:** Transformer-alapú étterem-, kávézó- és brunchhely-ajánlórendszer  
**Adat:** SemEval-2016 Task 5 – English Restaurants, Subtask 1

Ebben a notebookban az első tényleges Transformer-modellt készítjük el.

### Cél

Egy mondat alapján automatikusan felismerni, hogy **milyen aspektusokról beszél**.

Például:

> *“The food was excellent, but the service was slow.”*

Elvárt címkék:

- `FOOD#QUALITY = 1`
- `SERVICE#GENERAL = 1`
- minden más aspektus = 0

Mivel egy mondatban egyszerre több aspektus is szerepelhet, ez **multi-label classification** feladat.

### A notebook fő lépései

1. környezet és GPU ellenőrzése;
2. XML-adatok betöltése;
3. az `aspect_df` újraépítése;
4. review-szintű train / validation split;
5. DistilBERT tokenizer és modell betöltése;
6. tokenizálás;
7. finomhangolás;
8. validation értékelés precision / recall / F1 metrikákkal;
9. néhány konkrét predikció megtekintése.

A hivatalos gold test adathalmazt ebben a notebookban **még nem használjuk modellértékelésre**.


## 1. Szükséges csomagok telepítése

A Google Colab alapból nem feltétlenül ugyanazokat a Hugging Face csomagverziókat tartalmazza, ezért a szükséges könyvtárakat explicit telepítjük.

A fő csomagok:

- `transformers`: előtanított Transformer-modellek;
- `datasets`: Hugging Face adathalmaz-kezelés;
- `accelerate`: GPU-s tanítás támogatása;
- `scikit-learn`: train/validation split és értékelési metrikák.


In [ ]:
!pip -q install "transformers>=4.46,<5" datasets accelerate scikit-learn


In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    hamming_loss,
)

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 30)

RANDOM_STATE = 42

print("PyTorch verzió:", torch.__version__)
print("GPU elérhető:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


### GPU beállítása Colabban

Ehhez a notebookhoz már érdemes GPU-t használni.

Colabban:

**Futtatókörnyezet → Futtatókörnyezet típusának módosítása → T4 GPU**

Ha a fenti cellában a `GPU elérhető:` értéke `True`, akkor minden rendben.


## 2. Training XML feltöltése

Ebben a notebookban a modell fejlesztéséhez **csak a hivatalos training fájlt** használjuk:

- `ABSA16_Restaurants_Train_SB1_v2.xml`

A gold testet most szándékosan nem használjuk. Így elkerüljük, hogy a modellbeállításokat a végső teszteredmény alapján hangoljuk.


In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Nem Google Colab környezetben fut a notebook – a kézi feltöltést kihagyjuk.")


In [ ]:
TRAIN_PATH = Path("ABSA16_Restaurants_Train_SB1_v2.xml")

assert TRAIN_PATH.exists(), f"Hiányzó fájl: {TRAIN_PATH}"

print("A training XML-fájl elérhető.")


## 3. Az aspektus-adathalmaz újraépítése

A második notebookot szándékosan úgy készítjük, hogy önállóan is futtatható legyen.

Ezért röviden újra:

1. beolvassuk az XML-t;
2. mondatszintű táblát készítünk;
3. minden aspektushoz 0/1 címkét hozunk létre;
4. az `OutOfScope` mondatokat kizárjuk.

A változóneveket angolul tartjuk, mert közvetlenül követik a SemEval adatmezőit.


In [ ]:
def parse_semeval_xml(path):
    root = ET.parse(path).getroot()

    sentences = []
    opinions = []

    for review in root.findall("Review"):
        review_id = review.attrib["rid"]

        sentences_node = review.find("sentences")
        if sentences_node is None:
            continue

        for sentence in sentences_node.findall("sentence"):
            sentence_id = sentence.attrib["id"]
            text = sentence.findtext("text", default="")
            out_of_scope = sentence.attrib.get("OutOfScope", "FALSE").upper() == "TRUE"

            sentences.append({
                "review_id": review_id,
                "sentence_id": sentence_id,
                "text": text,
                "out_of_scope": out_of_scope,
            })

            opinions_node = sentence.find("Opinions")
            if opinions_node is None:
                continue

            for opinion in opinions_node.findall("Opinion"):
                opinions.append({
                    "review_id": review_id,
                    "sentence_id": sentence_id,
                    "text": text,
                    "target": opinion.attrib.get("target"),
                    "category": opinion.attrib.get("category"),
                    "polarity": opinion.attrib.get("polarity"),
                    "out_of_scope": out_of_scope,
                })

    return pd.DataFrame(sentences), pd.DataFrame(opinions)


sentences_df, opinions_df = parse_semeval_xml(TRAIN_PATH)

ASPECTS = sorted(opinions_df["category"].unique())

print("Mondatok száma:", len(sentences_df))
print("Opinion annotációk:", len(opinions_df))
print("Aspektuskategóriák száma:", len(ASPECTS))
print("\nAspektusok:")
for aspect in ASPECTS:
    print("-", aspect)


In [ ]:
def build_aspect_df(sentences_df, opinions_df, aspects):
    df = sentences_df.loc[~sentences_df["out_of_scope"]].copy()

    sentence_aspects = (
        opinions_df.loc[~opinions_df["out_of_scope"]]
        .groupby("sentence_id")["category"]
        .apply(set)
        .to_dict()
    )

    for aspect in aspects:
        df[aspect] = df["sentence_id"].map(
            lambda sid: int(aspect in sentence_aspects.get(sid, set()))
        )

    return df.reset_index(drop=True)


aspect_df = build_aspect_df(sentences_df, opinions_df, ASPECTS)

print("Modellezhető mondatok:", len(aspect_df))
display(aspect_df.head())


## 4. Review-szintű train / validation split

A hivatalos training adatot két részre bontjuk:

- **train:** kb. 80%
- **validation:** kb. 20%

### Miért review-szinten?

Ha ugyanannak a review-nak egyes mondatai trainbe, más mondatai validationbe kerülnének, a két rész túl hasonló szövegeket tartalmazhatna.

Ez **data leakage**-et okozhatna.

Ezért a `review_id` alapján csoportosítunk: egy review összes mondata mindig ugyanabba a részhalmazba kerül.


In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_idx, val_idx = next(
    splitter.split(
        aspect_df,
        groups=aspect_df["review_id"],
    )
)

train_df = aspect_df.iloc[train_idx].reset_index(drop=True)
val_df = aspect_df.iloc[val_idx].reset_index(drop=True)

print("Train mondatok:", len(train_df))
print("Validation mondatok:", len(val_df))

print("\nTrain review-k:", train_df["review_id"].nunique())
print("Validation review-k:", val_df["review_id"].nunique())

overlap = set(train_df["review_id"]) & set(val_df["review_id"])
print("\nÁtfedő review ID-k száma:", len(overlap))

assert len(overlap) == 0


### Címkeeloszlások ellenőrzése

A split csoportszinten történik, ezért nem garantált, hogy minden aspektus pontosan ugyanilyen arányban jelenik meg a két részben.

Ellenőrizzük, hogy nincs-e extrém eltérés.


In [ ]:
distribution = pd.DataFrame({
    "train_count": train_df[ASPECTS].sum(),
    "train_ratio": train_df[ASPECTS].mean(),
    "val_count": val_df[ASPECTS].sum(),
    "val_ratio": val_df[ASPECTS].mean(),
})

distribution["ratio_diff"] = (
    distribution["train_ratio"] - distribution["val_ratio"]
).abs()

display(distribution.sort_values("train_count", ascending=False))


## 5. A modellezési input előkészítése

A Hugging Face `Dataset` számára két fő mezőt készítünk:

- `text`: a mondat;
- `labels`: 12 elemű 0/1 vektor.

Például:

`[0, 0, 1, 0, 0, ..., 1]`

Ez azt jelenti, hogy az adott mondatban két aspektus aktív.


In [ ]:
def prepare_for_hf(df, aspects):
    result = pd.DataFrame()
    result["text"] = df["text"]

    result["labels"] = df[aspects].apply(
        lambda row: [float(x) for x in row.values],
        axis=1,
    )

    return result


train_hf_df = prepare_for_hf(train_df, ASPECTS)
val_hf_df = prepare_for_hf(val_df, ASPECTS)

display(train_hf_df.head())


## 6. DistilBERT tokenizer és modell

Első modellként a:

`distilbert-base-uncased`

előtanított modellt használjuk.

### Miért DistilBERT?

- BERT-típusú encoder modell;
- szövegértési / classification feladatokra alkalmas;
- kisebb és gyorsabb, mint a teljes BERT;
- Colab T4 GPU-n kényelmesen finomhangolható ezen a kis adathalmazon.

A modell végére egy 12 kimenetű classification head kerül.

Fontos: `problem_type="multi_label_classification"`, ezért a modell a címkéket **egymástól független bináris döntésekként** kezeli, és sigmoid-alapú BCE loss-t használ.


In [ ]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(ASPECTS),
    problem_type="multi_label_classification",
    id2label={i: label for i, label in enumerate(ASPECTS)},
    label2id={label: i for i, label in enumerate(ASPECTS)},
)

print("Modell:", MODEL_NAME)
print("Kimeneti címkék száma:", model.config.num_labels)


## 7. Tokenizálás

A Transformer nem közvetlenül szöveggel dolgozik.

A tokenizer a mondatokat tokenekre bontja, majd számsorozattá alakítja.

Most maximum **128 tokent** engedünk mondatonként. A SemEval mondatokhoz ez bőségesen elegendő, miközben gyorsabbá teszi a tanítást.


In [ ]:
train_dataset = Dataset.from_pandas(
    train_hf_df,
    preserve_index=False,
)

val_dataset = Dataset.from_pandas(
    val_hf_df,
    preserve_index=False,
)


def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
    )


train_tokenized = train_dataset.map(
    tokenize_batch,
    batched=True,
)

val_tokenized = val_dataset.map(
    tokenize_batch,
    batched=True,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_tokenized)
print(val_tokenized)


## 8. Értékelési metrikák

A modell minden aspektusra egy logitet ad.

A logiteket sigmoid függvénnyel 0 és 1 közötti valószínűséggé alakítjuk, majd kezdetben:

**0.5 felett → aspektus jelen van**

A fő metrikák:

### Micro precision / recall / F1
Az összes aspektusdöntést együtt kezeli. A gyakori aspektusok nagyobb hatással vannak rá.

### Macro precision / recall / F1
Minden aspektus teljesítményét külön kiszámolja, majd átlagolja. Emiatt a ritka aspektusok is ugyanakkora súlyt kapnak.

### Exact-match accuracy
Csak akkor számít helyesnek egy mondatot, ha **mind a 12 címke pontosan jó**.

### Hamming loss
A hibás 0/1 címkedöntések aránya. Minél kisebb, annál jobb.

A projekt témabejelentőjében vállalt precision / recall / F1 metrikákat tehát közvetlenül megkapjuk.


In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probabilities = sigmoid(logits)
    predictions = (probabilities >= 0.5).astype(int)
    labels = labels.astype(int)

    p_micro, r_micro, f1_micro, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="micro",
        zero_division=0,
    )

    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    exact_accuracy = accuracy_score(labels, predictions)
    hamming = hamming_loss(labels, predictions)

    return {
        "precision_micro": p_micro,
        "recall_micro": r_micro,
        "f1_micro": f1_micro,
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f1_macro,
        "exact_match_accuracy": exact_accuracy,
        "hamming_loss": hamming,
    }


## 9. Tanítási beállítások

Első körben szándékosan egyszerű és visszafogott konfigurációval indulunk:

- 3 epoch;
- batch size = 16;
- learning rate = 2e-5;
- weight decay = 0.01;
- minden epoch végén validation;
- a legjobb micro-F1 modellt tartjuk meg.

Ez az első **baseline Transformer-modellünk**. Nem akarunk még hyperparameter tuningot végezni.


In [ ]:
training_args = TrainingArguments(
    output_dir="aspect_distilbert_output",

    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    greater_is_better=True,

    save_total_limit=1,

    logging_steps=20,

    fp16=torch.cuda.is_available(),

    report_to="none",

    seed=RANDOM_STATE,
)

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,

    data_collator=data_collator,

    compute_metrics=compute_metrics,
)

print("A Trainer elkészült.")


## 10. Modell finomhangolása

A következő cella indítja el a tényleges tanítást.

T4 GPU-val ezen az adathalmazon kezelhető méretű futásra számíthatunk.

A tanítás során figyeld:

- `loss`: training loss;
- `eval_loss`: validation loss;
- `eval_f1_micro`;
- `eval_f1_macro`.

Ha a training loss csökken, de a validation teljesítmény romlik, az overfitting jele lehet.


In [ ]:
train_result = trainer.train()


## 11. Validation eredmények

Most külön is lefuttatjuk az értékelést a validation adathalmazon.

Ezek lesznek az első valódi Transformer-eredményeink.


In [ ]:
val_metrics = trainer.evaluate()

print("Validation eredmények:")
for key, value in val_metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")


## 12. Aspektusonkénti teljesítmény

A macro-F1 megmutatja, hogy a ritka aspektusok mennyire húzzák lefelé a teljesítményt, de hasznos látni az egyes címkéket külön is.

Ez a tábla megmutatja:

- precision;
- recall;
- F1;
- support = hány pozitív validation példa tartozik az aspektushoz.


In [ ]:
pred_output = trainer.predict(val_tokenized)

val_logits = pred_output.predictions
val_labels = pred_output.label_ids.astype(int)

val_probs = sigmoid(val_logits)
val_preds = (val_probs >= 0.5).astype(int)

p, r, f1, support = precision_recall_fscore_support(
    val_labels,
    val_preds,
    average=None,
    zero_division=0,
)

per_aspect_metrics = pd.DataFrame({
    "aspect": ASPECTS,
    "precision": p,
    "recall": r,
    "f1": f1,
    "support": support,
}).sort_values("support", ascending=False)

display(per_aspect_metrics)


## 13. Konkrét predikciók megtekintése

A metrikák mellett mindig érdemes kézzel is megnézni néhány példát.

Ez segít megérteni:

- milyen aspektusokat talál el a modell;
- mit hagy ki;
- milyen téves aspektusokat jelöl meg.


In [ ]:
def labels_from_vector(vector, aspects):
    return [
        aspect
        for aspect, value in zip(aspects, vector)
        if int(value) == 1
    ]


examples = val_df[["text"]].copy()

examples["true_aspects"] = [
    labels_from_vector(row, ASPECTS)
    for row in val_labels
]

examples["predicted_aspects"] = [
    labels_from_vector(row, ASPECTS)
    for row in val_preds
]

examples["correct"] = (
    examples["true_aspects"].astype(str)
    == examples["predicted_aspects"].astype(str)
)

display(examples.sample(
    min(15, len(examples)),
    random_state=RANDOM_STATE,
))


## 14. Saját mondatok kipróbálása

A következő függvénnyel tetszőleges új angol mondatra megnézhetjük a modell aspektuspredikcióit.


In [ ]:
def predict_aspects(text, threshold=0.5):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        logits = model(**inputs).logits[0]

    probs = torch.sigmoid(logits).cpu().numpy()

    results = pd.DataFrame({
        "aspect": ASPECTS,
        "probability": probs,
    })

    results["predicted"] = (
        results["probability"] >= threshold
    )

    return results.sort_values(
        "probability",
        ascending=False,
    )


predict_aspects(
    "The coffee was excellent, the atmosphere was cozy, but the service was very slow."
)


# 16. Baseline modell továbbfejlesztése

Az első DistilBERT-modell működő baseline-t adott, de a validation eredmények alapján két fontos problémát láttunk:

1. a `0.5`-ös alapértelmezett threshold túl szigorú volt;
2. az aspektuscímkék erősen kiegyensúlyozatlanok, ezért a modell főként a gyakori kategóriákat tanulta meg.

A továbbiakban ezt a két problémát külön kezeljük.

## Baseline összefoglaló

A 3. epoch végén, `threshold = 0.5` mellett:

- micro precision ≈ **0.813**
- micro recall ≈ **0.314**
- micro F1 ≈ **0.454**
- macro F1 ≈ **0.099**

A modell tehát pontos volt, amikor pozitív aspektust jelölt, de a valóban jelen lévő aspektusok jelentős részét kihagyta.


## 16.1. Threshold diagnosztika

Multi-label classification esetén a modell minden aspektushoz egy 0 és 1 közötti valószínűséget ad.

Az alapértelmezett szabály:

> `probability >= 0.5` → az aspektus jelen van.

Ez azonban nem feltétlenül optimális, ezért több lehetséges küszöbértéket kipróbálunk a validation adaton.


In [ ]:
# A baseline modell validation predikciói
pred_output = trainer.predict(val_tokenized)

val_logits = pred_output.predictions
val_labels = pred_output.label_ids.astype(int)

val_probs = sigmoid(val_logits)

threshold_results = []

for threshold in np.arange(0.05, 0.55, 0.05):
    preds = (val_probs >= threshold).astype(int)

    p_micro, r_micro, f1_micro, _ = precision_recall_fscore_support(
        val_labels,
        preds,
        average="micro",
        zero_division=0,
    )

    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        val_labels,
        preds,
        average="macro",
        zero_division=0,
    )

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision_micro": p_micro,
        "recall_micro": r_micro,
        "f1_micro": f1_micro,
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f1_macro,
    })

threshold_df = pd.DataFrame(threshold_results)

display(
    threshold_df.sort_values(
        "f1_micro",
        ascending=False
    )
)


### Baseline threshold választás

A validation adaton a legjobb micro-F1 körülbelül **0.30-as thresholdnál** adódott.

A fő eredmények:

- micro precision ≈ **0.688**
- micro recall ≈ **0.593**
- micro F1 ≈ **0.637**
- macro F1 ≈ **0.210**

Ez jelentős javulás a `0.5`-ös küszöbhöz képest.

A macro-F1 azonban továbbra is alacsony maradt, ami arra utalt, hogy a ritkább aspektusokat a modell még mindig gyengén kezeli.


## 16.2. Valószínűségek aspektusonként

Megvizsgáljuk, hogy az egyes aspektusokhoz milyen valószínűségeket rendel a baseline modell.

Ez segít eldönteni, hogy egy kategória csak a magas threshold miatt nem jelenik meg, vagy a modell valóban nem tanulta meg megfelelően.


In [ ]:
probability_diagnostics = pd.DataFrame({
    "aspect": ASPECTS,
    "mean_probability": val_probs.mean(axis=0),
    "max_probability": val_probs.max(axis=0),
    "positive_examples": val_labels.sum(axis=0),
})

display(
    probability_diagnostics.sort_values(
        "positive_examples",
        ascending=False
    )
)


## 16.3. Baseline teljesítmény `threshold = 0.30` mellett


In [ ]:
BASELINE_THRESHOLD = 0.30

val_preds_030 = (
    val_probs >= BASELINE_THRESHOLD
).astype(int)

p, r, f1, support = precision_recall_fscore_support(
    val_labels,
    val_preds_030,
    average=None,
    zero_division=0,
)

per_aspect_metrics_030 = pd.DataFrame({
    "aspect": ASPECTS,
    "precision": p,
    "recall": r,
    "f1": f1,
    "support": support,
}).sort_values(
    "support",
    ascending=False
)

display(per_aspect_metrics_030)


### Következtetés a baseline modellről

A threshold tuning után a gyakori aspektusok teljesítménye már megfelelő volt, például:

- `FOOD#QUALITY`
- `SERVICE#GENERAL`
- `RESTAURANT#GENERAL`
- `AMBIENCE#GENERAL`

A ritkább aspektusok jelentős részénél azonban továbbra is `0` körüli F1 jelent meg.

Ez arra utalt, hogy a fő probléma már nem egyszerűen a threshold, hanem az **osztály-aránytalanság (class imbalance)**.


# 17. Class-weighted DistilBERT

A ritka aspektusok kezelése érdekében súlyozott veszteségfüggvényt használunk.

A multi-label classification alapértelmezett vesztesége a **Binary Cross Entropy with Logits Loss (BCEWithLogitsLoss)**.

A `pos_weight` paraméterrel nagyobb büntetést adunk akkor, ha a modell egy ritka pozitív példát kihagy.

Az egyes aspektusok súlya:

\[
pos\_weight_j = \frac{N - P_j}{P_j}
\]

ahol:

- \(N\): training mondatok száma;
- \(P_j\): az adott aspektus pozitív példáinak száma.

Minél ritkább egy aspektus, annál nagyobb súlyt kap.


In [ ]:
positive_counts = train_df[ASPECTS].sum(axis=0).values
negative_counts = len(train_df) - positive_counts

pos_weights = negative_counts / positive_counts

pos_weight_tensor = torch.tensor(
    pos_weights,
    dtype=torch.float32
)

class_weight_table = pd.DataFrame({
    "aspect": ASPECTS,
    "positive_examples": positive_counts,
    "pos_weight": pos_weights
}).sort_values("positive_examples")

display(class_weight_table)


## 17.1. Saját WeightedTrainer

A Hugging Face `Trainer` alapértelmezésben nem használ címkénként eltérő `pos_weight` értékeket, ezért egy rövid saját Trainert készítünk.

A modellezési pipeline minden más része változatlan marad, így a baseline és a weighted modell tisztán összehasonlítható.


In [ ]:
from torch.nn import BCEWithLogitsLoss


class WeightedTrainer(Trainer):

    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = BCEWithLogitsLoss(
            pos_weight=self.pos_weight.to(logits.device)
        )

        loss = loss_fn(
            logits,
            labels.float()
        )

        return (loss, outputs) if return_outputs else loss


## 17.2. Új modell inicializálása

Fontos, hogy a weighted modellt **nem a már finomhangolt baseline modellből** indítjuk.

Újra az eredeti `distilbert-base-uncased` modellből indulunk, így a két kísérlet közötti fő különbség valóban a veszteségfüggvény súlyozása.


In [ ]:
weighted_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(ASPECTS),
    problem_type="multi_label_classification",
    id2label={i: label for i, label in enumerate(ASPECTS)},
    label2id={label: i for i, label in enumerate(ASPECTS)},
)


In [ ]:
weighted_trainer = WeightedTrainer(
    model=weighted_model,
    args=training_args,

    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,

    data_collator=data_collator,
    compute_metrics=compute_metrics,

    pos_weight=pos_weight_tensor,
)

print("A weighted Trainer elkészült.")


## 17.3. Weighted modell finomhangolása

Ugyanazokat a fő tanítási beállításokat használjuk, mint a baseline esetén:

- 3 epoch;
- azonos train / validation split;
- azonos learning rate;
- azonos batch size;
- azonos random seed.

Így a két modell eredménye közvetlenül összehasonlítható.


In [ ]:
weighted_train_result = weighted_trainer.train()


# 18. Weighted modell threshold tuning

A weighted loss miatt a modell valószínűségi eloszlása megváltozik, ezért a baseline `0.30` thresholdját nem használjuk automatikusan.

Új threshold sweepet végzünk a validation adaton.


In [ ]:
weighted_pred_output = weighted_trainer.predict(val_tokenized)

weighted_logits = weighted_pred_output.predictions
weighted_labels = weighted_pred_output.label_ids.astype(int)

weighted_probs = sigmoid(weighted_logits)

weighted_threshold_results = []

for threshold in np.arange(0.05, 0.85, 0.05):

    preds = (weighted_probs >= threshold).astype(int)

    p_micro, r_micro, f1_micro, _ = precision_recall_fscore_support(
        weighted_labels,
        preds,
        average="micro",
        zero_division=0,
    )

    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        weighted_labels,
        preds,
        average="macro",
        zero_division=0,
    )

    weighted_threshold_results.append({
        "threshold": round(threshold, 2),
        "precision_micro": p_micro,
        "recall_micro": r_micro,
        "f1_micro": f1_micro,
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f1_macro,
    })

weighted_threshold_df = pd.DataFrame(
    weighted_threshold_results
)

display(
    weighted_threshold_df.sort_values(
        "f1_micro",
        ascending=False
    )
)


## 18.1. Végleges threshold választása

A validation adaton a weighted modell legjobb kompromisszumát:

**`threshold = 0.55`**

adta.

Körülbelüli eredmények:

- micro precision ≈ **0.469**
- micro recall ≈ **0.729**
- micro F1 ≈ **0.570**
- macro F1 ≈ **0.502**

### Baseline vs. weighted modell

| Modell | Threshold | Micro F1 | Macro F1 |
|---|---:|---:|---:|
| Baseline DistilBERT | 0.30 | ~0.637 | ~0.210 |
| Weighted DistilBERT | 0.55 | ~0.570 | ~0.502 |

A baseline jobb micro-F1-et ad, mert főként a gyakori címkéken teljesít jól.

A weighted modell viszont sokkal kiegyensúlyozottabb teljesítményt ad a különböző aspektusokon. A recommendation use case miatt ezt választjuk végleges modellnek, mert fontos, hogy ne csak a leggyakoribb kategóriákat legyen képes felismerni.


## 18.2. Aspektusonkénti teljesítmény a végleges weighted modellen


In [ ]:
FINAL_ASPECT_THRESHOLD = 0.55

weighted_preds_final = (
    weighted_probs >= FINAL_ASPECT_THRESHOLD
).astype(int)

p, r, f1, support = precision_recall_fscore_support(
    weighted_labels,
    weighted_preds_final,
    average=None,
    zero_division=0,
)

weighted_per_aspect = pd.DataFrame({
    "aspect": ASPECTS,
    "precision": p,
    "recall": r,
    "f1": f1,
    "support": support,
}).sort_values(
    "support",
    ascending=False
)

display(weighted_per_aspect)


### Az eredmények értelmezése

A fontosabb, recommendation szempontból hasznos kategóriák már értelmezhetően működnek:

- `FOOD#QUALITY`
- `SERVICE#GENERAL`
- `RESTAURANT#GENERAL`
- `AMBIENCE#GENERAL`
- `DRINKS#QUALITY`
- `FOOD#PRICES`
- `RESTAURANT#PRICES`

A nagyon ritka kategóriáknál az F1 értékeket óvatosan kell értelmezni, mert a validation support rendkívül alacsony lehet.

Például néhány kategóriában csak néhány, vagy akár egyetlen validation példa található. Ezekből nem lehet stabil általánosító következtetést levonni.


# 19. A végleges aspect detector mentése

Most már nem a baseline `trainer`, hanem a **weighted_trainer** modelljét mentjük.

A végleges globális thresholdot (`0.55`) külön változóban is rögzítjük, mert ezt a későbbi recommendation pipeline-ban is használni fogjuk.


In [ ]:
FINAL_ASPECT_MODEL_PATH = "aspect_distilbert_weighted"
FINAL_ASPECT_THRESHOLD = 0.55

weighted_trainer.save_model(FINAL_ASPECT_MODEL_PATH)
tokenizer.save_pretrained(FINAL_ASPECT_MODEL_PATH)

print("Weighted aspect modell elmentve:", FINAL_ASPECT_MODEL_PATH)
print("Végleges threshold:", FINAL_ASPECT_THRESHOLD)


# 20. Végleges predikciós függvény

Ezzel a függvénnyel tetszőleges új angol mondat aspektusait előre jelezhetjük.

A függvény már:

- a weighted modellt használja;
- a végleges `0.55` thresholdot használja;
- minden aspektushoz visszaadja a valószínűséget.


In [ ]:
def predict_aspects_final(
    text,
    threshold=FINAL_ASPECT_THRESHOLD
):
    weighted_model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    )

    inputs = {
        key: value.to(weighted_model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        logits = weighted_model(**inputs).logits[0]

    probs = torch.sigmoid(logits).cpu().numpy()

    results = pd.DataFrame({
        "aspect": ASPECTS,
        "probability": probs,
    })

    results["predicted"] = (
        results["probability"] >= threshold
    )

    return results.sort_values(
        "probability",
        ascending=False
    )


In [ ]:
predict_aspects_final(
    "The coffee was excellent, the atmosphere was cozy, but the service was very slow."
)


# 21. Aspect Detection – végső összefoglalás

Ebben a notebookban két DistilBERT-alapú modellt hasonlítottunk össze.

## 1. Baseline modell

- standard multi-label BCE loss;
- `threshold = 0.30` validation tuning után;
- magasabb micro-F1;
- főként a gyakori aspektusokat tanulta meg jól.

## 2. Class-weighted modell

- weighted BCE loss;
- `threshold = 0.55`;
- valamivel alacsonyabb micro-F1;
- lényegesen jobb macro-F1;
- több ritka aspektus felismerésére is képes.

### Végleges döntés

A projekt további részeihez a:

**Class-weighted DistilBERT + 0.55 threshold**

modellt használjuk.

Ennek oka, hogy a későbbi restaurant profile és recommendation rendszer számára az aspektusok **szélesebb lefedettsége** fontosabb, mint az, hogy kizárólag a leggyakoribb kategóriákon maximalizáljuk a teljesítményt.

## Következő komponens

A következő notebook az:

**Aspect-Level Sentiment Classification**

feladatot valósítja meg:

**sentence + target + aspect → positive / neutral / negative**


# 22. Végleges modell ZIP-be csomagolása és letöltése

A Colab runtime nem tartós, ezért a finomhangolt aspect modellt ZIP fájlba csomagoljuk és letöltjük.

Ezt a ZIP-et fogjuk majd feltölteni az **05 – Place Profile Generation** notebookba.


In [ ]:
import shutil

ASPECT_ZIP_BASE = "aspect_distilbert_weighted"

shutil.make_archive(
    ASPECT_ZIP_BASE,
    "zip",
    FINAL_ASPECT_MODEL_PATH
)

print("Elkészült:", ASPECT_ZIP_BASE + ".zip")

try:
    from google.colab import files
    files.download(ASPECT_ZIP_BASE + ".zip")
except ImportError:
    print("A ZIP a notebook mappájában található.")
